In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 · Data quality gate
# MAGIC Last task in the job. Runs assertions across the layers and **raises an error if any fail**,
# MAGIC which marks the job run as failed and sends the failure notification.
# MAGIC
# MAGIC The thresholds are set from what the source normally looks like (about 0.5% malformed lines).
# MAGIC A sudden jump means something changed upstream, and someone should look before the dashboard is trusted.

# COMMAND ----------

C = "incident_analytics"


def scalar(sql: str):
    return spark.sql(sql).collect()[0][0]


checks = [
    # (name, value, passes?)
    ("silver: duplicate event_ids",
     v := scalar(f"SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM {C}.silver.app_logs"), v == 0),

    ("silver: rows with NULL event_ts",
     v := scalar(f"SELECT COUNT(*) FROM {C}.silver.app_logs WHERE event_ts IS NULL"), v == 0),

    ("silver: unexpected log levels",
     v := scalar(f"SELECT COUNT(*) FROM {C}.silver.app_logs WHERE level NOT IN ('INFO','WARN','ERROR')"), v == 0),

    ("reconciliation: bronze rows unaccounted for (must be >= 0)",
     v := scalar(f"""SELECT (SELECT COUNT(*) FROM {C}.bronze.app_logs_raw)
                          - (SELECT COUNT(*) FROM {C}.silver.app_logs)
                          - (SELECT COUNT(*) FROM {C}.silver.app_logs_quarantine)"""), v >= 0),

    ("quarantine rate % (threshold 2%)",
     v := scalar(f"""SELECT ROUND(100 * (SELECT COUNT(*) FROM {C}.silver.app_logs_quarantine)
                                / (SELECT COUNT(*) FROM {C}.bronze.app_logs_raw), 2)"""), v <= 2),

    ("incidents: resolved before opened",
     v := scalar(f"SELECT COUNT(*) FROM {C}.silver.incidents WHERE resolved_at < opened_at"), v == 0),

    ("incidents: pointing at an unknown deployment",
     v := scalar(f"""SELECT COUNT(*) FROM {C}.silver.incidents i
                     LEFT ANTI JOIN {C}.silver.deployments d ON d.deployment_id = i.related_deployment_id"""), v == 0),

    ("gold: incidents whose deploy hour had no spike",
     v := scalar(f"""SELECT COUNT(*) FROM {C}.gold.deployment_impact
                     WHERE caused_incident AND NOT spike_in_deploy_hour"""), v == 0),

    ("freshness: hours since last Bronze ingest (threshold 6)",
     v := scalar(f"""SELECT ROUND(timestampdiff(MINUTE, MAX(_ingested_at), current_timestamp()) / 60.0, 1)
                     FROM {C}.bronze.app_logs_raw"""), v is not None and v <= 6),
]

# COMMAND ----------

for name, value, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {value}")

failed = [name for name, _, ok in checks if not ok]
if failed:
    raise AssertionError(f"{len(failed)} data quality check(s) failed: {failed}")
print(f"\nAll {len(checks)} checks passed.")